# Evaluate one final benchmark run

Use the same model environment as training. The notebook discovers the latest compatible completed final run and evaluates official validation.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


## Configuration


In [ ]:
MODEL_ID = "rtdetrv2_l"
DATASET_TRACK = "2class"
RUN_ID = ""  # Leave blank unless the discovery table shows multiple compatible runs.
EVALUATION_RESOLUTION = 640


## Restore and validate the selected model environment


In [ ]:
if IS_COLAB and not SMOKE_TEST:
    requirement = (
        "requirements-rtdetr-colab.txt"
        if MODEL_ID == "rtdetrv2_l"
        else "requirements-openmmlab-py310-cu118.txt"
    )
    if MODEL_ID != "rtdetrv2_l" and sys.version_info[:2] != (3, 10):
        raise RuntimeError("Use the documented Python 3.10 custom/local runtime.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", requirement],
        check=True,
    )
if MODEL_ID != "rtdetrv2_l":
    upstream = Path("/content/VMamba" if MODEL_ID == "faster_rcnn_vmamba_t" else "/content/mmdetection")
    if IS_COLAB and not upstream.joinpath(".git").is_dir() and not SMOKE_TEST:
        url = (
            "https://github.com/MzeroMiko/VMamba.git"
            if MODEL_ID == "faster_rcnn_vmamba_t"
            else "https://github.com/open-mmlab/mmdetection.git"
        )
        clone_command = ["git", "clone"]
        if MODEL_ID != "faster_rcnn_vmamba_t":
            clone_command.extend(["--depth", "1", "--branch", "v3.3.0"])
        clone_command.extend([url, str(upstream)])
        subprocess.run(clone_command, check=True)
    os.environ[
        "VMAMBA_ROOT" if MODEL_ID == "faster_rcnn_vmamba_t" else "MMDET_ROOT"
    ] = str(upstream)
    if MODEL_ID == "faster_rcnn_vmamba_t":
        subprocess.run(
            ["git", "-C", str(upstream), "checkout",
             "2ed52ead062a51a64521ed3871d52914bf532876"],
            check=True,
        )
        os.environ["VMAMBA_T_PRETRAINED"] = str(paths.pretrained / "vmamba_t.pth")
        if not Path(os.environ["VMAMBA_T_PRETRAINED"]).is_file() and not SMOKE_TEST:
            raise FileNotFoundError(os.environ["VMAMBA_T_PRETRAINED"])
        if not SMOKE_TEST:
            try:
                import selective_scan_cuda
            except ImportError:
                subprocess.run(
                    [sys.executable, "-m", "pip", "install",
                     str(upstream / "kernels" / "selective_scan"),
                     "--no-build-isolation"],
                    check=True,
                )
from src.notebook_utils import require_gpu, require_model_environment
if not SMOKE_TEST:
    require_model_environment("rtdetr" if MODEL_ID == "rtdetrv2_l" else "openmmlab")
    require_gpu(MODEL_ID)
print("Model environment preflight:", "SMOKE_TEST" if SMOKE_TEST else "PASS")


## Discover the final run


In [ ]:
import pandas as pd
from src.training.checkpointing import RunRegistry
from src.utils.serialization import read_yaml

registry = RunRegistry(paths)
candidates = []
for run in registry.list_available_runs(MODEL_ID, DATASET_TRACK, status="completed"):
    run_dir = Path(run.get("run_dir") or paths.final_checkpoints / MODEL_ID / run["run_id"])
    training_config = run_dir / "training_config.yaml"
    if training_config.exists() and read_yaml(training_config).get("run_kind") == "final_complete_official_train":
        candidates.append({**run, "run_dir": str(run_dir)})
display(pd.DataFrame([
    {key: row.get(key) for key in ("run_id", "created_at", "status", "best_validation_map", "run_dir")}
    for row in candidates
]))
if RUN_ID:
    selected_runs = [row for row in candidates if row["run_id"] == RUN_ID]
elif len(candidates) == 1:
    selected_runs = candidates
elif len(candidates) > 1:
    raise RuntimeError("Multiple compatible final runs found. Copy one run_id into RUN_ID and rerun.")
elif SMOKE_TEST:
    selected_runs = []
else:
    raise RuntimeError("No compatible completed final run. Finish notebook 13 first.")
SELECTED_RUN_ID = selected_runs[0]["run_id"] if selected_runs else None
print("Selected final run:", SELECTED_RUN_ID or "SMOKE_TEST: none required")


## Evaluate complete official validation


In [ ]:
if SMOKE_TEST:
    print("SMOKE_TEST: final-run discovery passed; GPU evaluation was skipped.")
else:
    command = [
        sys.executable, "scripts/evaluate.py",
        "--drive-root", DRIVE_ROOT,
        "--dataset-track", DATASET_TRACK,
        "--split", "val",
        "--run-id", SELECTED_RUN_ID,
        "--resolutions", str(EVALUATION_RESOLUTION),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
    evaluation_files = sorted(paths.evaluation.glob(f"{SELECTED_RUN_ID}__res*__metrics.json"))
    if not evaluation_files:
        raise RuntimeError("Evaluation completed without a metrics file.")
    recommended = f"{MODEL_ID}__{DATASET_TRACK}__" + __import__("datetime").datetime.now(
        __import__("datetime").timezone.utc
    ).strftime("%Y%m%d_%H%M%S")
    print("\nRESULTS READY FOR REVIEW")
    print("\nEvaluation directory:", paths.evaluation)
    print("Report directory:", paths.reports)
    print("Recommended result bundle ID:", recommended)
    print("Next notebook:", REPO_DIR / "notebooks" / "10_generate_final_report.ipynb")
